# Strategy 2 (E2-B) — Results

**Pipeline:** ICD-10 knowledge graph + embedding propagation + C×S discriminative criterion + lexical top-N prefiltration + semantic recall against the reference terminology.

This notebook reproduces the results and ablations of **Section 5.2** of the thesis. All logic lives in the `asthma_e2` package; the notebook only orchestrates and visualises.

**Sections**
- 0. Imports and configuration
- 1. Reference terminology
- 2. Data loading
- 3. Reference term embeddings
- 4. Graph construction and propagation *(5.2.1)*
- 5. C×S criterion and discriminative subspace
- 6. Reference pipeline *(5.2.2)*
- 7. Graph ablation *(5.2.3)*
- 8. Propagation ablation *(5.2.4)*
- 9. C×S margin ablation *(5.2.5)*
- 10. Discriminative subspace analysis *(5.2.6)*
- 11. Lexical prefiltration — top-N ablation *(5.2.7)*
- 12. System comparison *(5.2.8)*
- 13. Error analysis *(5.2.9)*
- 14. Computational cost *(5.2.10)*

## 0. Imports and configuration

In [ ]:
import sys, time, itertools
from pathlib import Path

# Make the src/ package importable when running from notebooks/.
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, precision_score, recall_score,
                             f1_score, precision_recall_curve)

from asthma_e2 import (CONFIG, PATHS, REFERENCE_TERMS, build_graph,
                       propagate_embeddings, compute_CS, cv_evaluate)
from asthma_e2.data_loading import (load_icd10_catalogue, load_icd10_embeddings,
                                    load_nhc_mapping, load_patient_data)
from asthma_e2.embeddings import embed_batch
from asthma_e2.scoring import asthma_centroid, score_all_patients

PATHS.figures_dir.mkdir(parents=True, exist_ok=True)
PATHS.tables_dir.mkdir(parents=True, exist_ok=True)
print('Config:', CONFIG)

## 1. Reference terminology

A small set of asthma-related Spanish clinical terms anchors the discriminative subspace and the patient score.

In [ ]:
print(f'Reference terms (m={len(REFERENCE_TERMS)}):')
for i, t in enumerate(REFERENCE_TERMS, 1):
    print(f'  {i:2d}. {t}')

## 2. Data loading

Loads the private artefacts from `data/` (see `data/README.md`). The clinical data are **not** distributed with this repository.

In [ ]:
df_icd10 = load_icd10_catalogue()
codes, emb_icd10 = load_icd10_embeddings()
nhc_mapping = load_nhc_mapping()
patients = load_patient_data(nhc_mapping=nhc_mapping)

labels_all = patients.labels
n_pos = int(labels_all.sum())
print(f'ICD-10 catalogue : {len(df_icd10):,} entries')
print(f'ICD-10 embeddings: {emb_icd10.shape[0]:,} codes x {emb_icd10.shape[1]} dims')
print(f'Patients         : {len(patients.ids)} (asthma={n_pos}, non-asthma={len(patients.ids)-n_pos})')

## 3. Reference term embeddings

In [ ]:
term_emb_raw = embed_batch(list(REFERENCE_TERMS))
centroid = asthma_centroid(term_emb_raw)
print(f'Term embeddings: {term_emb_raw.shape}')
print(f'Asthma centroid: {centroid.shape}')

## 4. Graph construction and propagation  *(5.2.1)*

In [ ]:
t0 = time.perf_counter()
G_ref, n_hier_ref, n_sem_ref = build_graph(
    df_icd10, emb_icd10, codes,
    threshold=CONFIG.graph_threshold, k_neighbors=CONFIG.graph_k_neighbors)
degrees = [d for _, d in G_ref.degree()]
t_graph = time.perf_counter() - t0

print(f'Nodes        : {G_ref.number_of_nodes():,}')
print(f'Hierarchical : {n_hier_ref:,}')
print(f'Semantic     : {n_sem_ref:,}')
print(f'Total edges  : {G_ref.number_of_edges():,}')
print(f'Mean degree  : {np.mean(degrees):.2f}')
print(f'Build time   : {t_graph:.1f} s')

table_1 = pd.DataFrame({
    'Metric': ['Nodes (ICD-10 codes)', 'Hierarchical edges', 'Semantic edges',
               'Total edges', 'Mean degree', 'Median degree', 'Isolated nodes'],
    'Value':  [G_ref.number_of_nodes(), n_hier_ref, n_sem_ref,
               G_ref.number_of_edges(), f'{np.mean(degrees):.2f}',
               int(np.median(degrees)), sum(1 for d in degrees if d == 0)],
})
table_1.to_csv(PATHS.tables_dir / 'table_1_graph.csv', index=False)
table_1

In [ ]:
t0 = time.perf_counter()
emb_icd10_prop = propagate_embeddings(
    G_ref, emb_icd10, codes,
    steps=CONFIG.propagation_steps, alpha=CONFIG.propagation_alpha, verbose=True)
t_prop = time.perf_counter() - t0
print(f'Propagation time: {t_prop:.1f} s')

## 5. C×S criterion and discriminative subspace

In [ ]:
score_D, C_ref, S_ref = compute_CS(term_emb_raw, emb_icd10_prop,
                                   margen=CONFIG.margin_gamma)
dims_ranked = np.argsort(score_D)[::-1]

print(f'C  mean:{C_ref.mean():.4f}  max:{C_ref.max():.4f}')
print(f'S  mean:{S_ref.mean():.4f}  max:{S_ref.max():.4f}  dims>0.5:{(S_ref>0.5).sum()}')
print(f'D  mean:{score_D.mean():.4f}  max:{score_D.max():.4f}')
for u in (0.50, 0.30, 0.20, 0.10):
    print(f'  dims D>{u:.2f}: {(score_D>u).sum()}')

## 6. Reference pipeline  *(5.2.2)*

In [ ]:
df_ref = cv_evaluate(score_D, term_emb_raw, patients,
                     umbral_criterio='f1')
cols = ['best_tau','n_dims','AUC','AUPR','Precision','Recall','F1','Specificity']
df_ref.to_csv(PATHS.tables_dir / 'table_2_reference_pipeline.csv')
df_ref[cols].round(4)

## 7. Graph ablation — threshold × k_neighbors  *(5.2.3)*

In [ ]:
THRESHOLDS = [0.65, 0.75, 0.85]
K_NEIGHBORS_LIST = [5, 10, 20]

rows_graph = []
for thr, k_n in tqdm(list(itertools.product(THRESHOLDS, K_NEIGHBORS_LIST)),
                     desc='Graph ablation'):
    G_tmp, _, n_sem = build_graph(df_icd10, emb_icd10, codes, thr, k_n)
    emb_tmp = propagate_embeddings(G_tmp, emb_icd10, codes,
                                   steps=CONFIG.propagation_steps,
                                   alpha=CONFIG.propagation_alpha)
    score_tmp, _, _ = compute_CS(term_emb_raw, emb_tmp, margen=CONFIG.margin_gamma)
    df_tmp = cv_evaluate(score_tmp, term_emb_raw, patients)
    rows_graph.append({'threshold': thr, 'k_neighbors': k_n,
                       'semantic_edges': n_sem,
                       'AUC_mean': df_tmp.loc['mean','AUC'],
                       'AUC_std':  df_tmp.loc['std','AUC'],
                       'F1_mean':  df_tmp.loc['mean','F1'],
                       'Recall_mean': df_tmp.loc['mean','Recall'],
                       'Precision_mean': df_tmp.loc['mean','Precision']})

table_3 = pd.DataFrame(rows_graph)
table_3.to_csv(PATHS.tables_dir / 'table_3_graph_ablation.csv', index=False)

pivot = table_3.pivot(index='threshold', columns='k_neighbors', values='AUC_mean')
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax,
            cbar_kws={'label': 'AUC (5-fold CV)'})
ax.set_title('Graph ablation: AUC vs threshold and k neighbours')
plt.tight_layout()
plt.savefig(PATHS.figures_dir / 'fig_3_graph_ablation.png', dpi=150)
plt.show()
table_3

## 8. Propagation ablation — steps × α  *(5.2.4)*

In [ ]:
STEPS_LIST = [0, 1, 2, 3]
ALPHAS = [0.3, 0.5, 0.7, 0.9]

rows_prop = []
for steps in STEPS_LIST:
    for alpha in (ALPHAS if steps > 0 else [None]):
        emb_tmp = (emb_icd10.copy() if steps == 0 else
                   propagate_embeddings(G_ref, emb_icd10, codes,
                                        steps=steps, alpha=alpha))
        score_tmp, _, _ = compute_CS(term_emb_raw, emb_tmp, margen=CONFIG.margin_gamma)
        df_tmp = cv_evaluate(score_tmp, term_emb_raw, patients)
        rows_prop.append({'steps': steps, 'alpha': alpha if steps > 0 else '-',
                          'AUC_mean': df_tmp.loc['mean','AUC'],
                          'AUC_std':  df_tmp.loc['std','AUC'],
                          'F1_mean':  df_tmp.loc['mean','F1'],
                          'Recall_mean': df_tmp.loc['mean','Recall']})

table_4 = pd.DataFrame(rows_prop)
table_4.to_csv(PATHS.tables_dir / 'table_4_propagation_ablation.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 4.5))
base = table_4[table_4['alpha'] == '-']['AUC_mean'].values[0]
for alpha in ALPHAS:
    sub = table_4[table_4['alpha'] == alpha]
    ax.plot([0] + sub['steps'].tolist(), [base] + sub['AUC_mean'].tolist(),
            'o-', lw=2, label=f'α={alpha}')
ax.set_xlabel('Propagation steps'); ax.set_ylabel('Mean AUC (5-fold CV)')
ax.set_title('Propagation ablation: AUC vs steps and α')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PATHS.figures_dir / 'fig_4_propagation_ablation.png', dpi=150)
plt.show()
table_4

## 9. C×S criterion ablation — margin γ  *(5.2.5)*

In [ ]:
MARGINS = [0.25, 0.50, 0.75, 1.00, 1.50, 2.00]
rows_cs = []
for margin in MARGINS:
    score_tmp, _, S_tmp = compute_CS(term_emb_raw, emb_icd10_prop, margen=margin)
    df_tmp = cv_evaluate(score_tmp, term_emb_raw, patients)
    rows_cs.append({'gamma': margin, 'dims_S_gt_05': int((S_tmp > 0.5).sum()),
                    'AUC_mean': df_tmp.loc['mean','AUC'],
                    'AUC_std':  df_tmp.loc['std','AUC'],
                    'F1_mean':  df_tmp.loc['mean','F1'],
                    'Precision_mean': df_tmp.loc['mean','Precision'],
                    'Recall_mean': df_tmp.loc['mean','Recall']})

table_5 = pd.DataFrame(rows_cs)
table_5.to_csv(PATHS.tables_dir / 'table_5_cs_ablation.csv', index=False)
table_5.round(4)

## 10. Discriminative subspace analysis  *(5.2.6)*

In [ ]:
auc_per_tau = []
for u in CONFIG.umbral_range:
    dims_u = np.where(score_D > u)[0]
    if len(dims_u) < 2:
        continue
    sc = score_all_patients(patients, term_emb_raw, dims_u, top_n=CONFIG.top_n_tokens)
    auc_per_tau.append({'tau': u, 'n_dims': len(dims_u),
                        'AUC': roc_auc_score(labels_all, sc)})

df_tau = pd.DataFrame(auc_per_tau)
df_tau.to_csv(PATHS.tables_dir / 'table_6_subspace.csv', index=False)

fig, ax1 = plt.subplots(figsize=(7, 4.5))
ax2 = ax1.twinx()
ax1.plot(df_tau['tau'], df_tau['AUC'], 'o-', color='#2ecc71', lw=2)
ax2.bar(df_tau['tau'], df_tau['n_dims'], width=0.03, alpha=0.3, color='#3498db')
ax1.set_xlabel('Threshold τ on D(d)')
ax1.set_ylabel('AUC (global)', color='#2ecc71')
ax2.set_ylabel('Number of selected dimensions', color='#3498db')
ax1.set_title('Discriminative subspace: AUC and dimension count vs τ')
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PATHS.figures_dir / 'fig_6_subspace_tau.png', dpi=150)
plt.show()
df_tau

## 11. Lexical prefiltration — top-N ablation  *(5.2.7)*

In [ ]:
dims_ref = np.where(score_D > 0.20)[0]
TOP_N_LIST = [10, 20, 30, 50, 75, 100, 200, 99999]  # 99999 = all tokens
rows_topn = []

for top_n in TOP_N_LIST:
    sc = score_all_patients(patients, term_emb_raw, dims_ref, top_n=top_n)
    auc = roc_auc_score(labels_all, sc)
    sep = sc[labels_all == 1].mean() - sc[labels_all == 0].mean()
    skf = StratifiedKFold(n_splits=CONFIG.cv_n_splits, shuffle=True,
                          random_state=CONFIG.random_state)
    fold_rows = []
    for tr, te in skf.split(sc, labels_all):
        p, r, thr = precision_recall_curve(labels_all[tr], sc[tr])
        f1 = 2*p*r/(p+r+1e-8)
        threshold = thr[min(np.argmax(f1[:-1]), len(thr)-1)] if len(thr) else sc[tr].mean()
        preds = (sc[te] >= threshold).astype(int)
        fold_rows.append({'F1': f1_score(labels_all[te], preds, zero_division=0),
                          'R': recall_score(labels_all[te], preds, zero_division=0),
                          'P': precision_score(labels_all[te], preds, zero_division=0)})
    df_f = pd.DataFrame(fold_rows)
    rows_topn.append({'top_N': str(top_n) if top_n < 99999 else 'all',
                      'AUC': auc, 'sep': sep,
                      'F1_mean': df_f['F1'].mean(),
                      'P_mean': df_f['P'].mean(), 'R_mean': df_f['R'].mean()})

table_7 = pd.DataFrame(rows_topn)
table_7.to_csv(PATHS.tables_dir / 'table_7_topn.csv', index=False)
table_7.round(4)

## 12. System comparison  *(5.2.8)*

E2-B at the F1-optimal threshold and at the recall=1.0 operating point, against the Strategy 1 baseline (rules + NegEx, values from Section 5.1.3).

In [ ]:
df_ref_r1 = cv_evaluate(score_D, term_emb_raw, patients,
                        umbral_criterio='recall1')

table_comp = pd.DataFrame([
    {'System': 'E1 (rules + NegEx)', 'AUC': float('nan'),
     'Precision': 0.87, 'Recall': 1.00, 'F1': 0.93, 'Specificity': 0.94},
    {'System': 'E2-B (F1-optimal)',
     'AUC': df_ref.loc['mean','AUC'], 'Precision': df_ref.loc['mean','Precision'],
     'Recall': df_ref.loc['mean','Recall'], 'F1': df_ref.loc['mean','F1'],
     'Specificity': df_ref.loc['mean','Specificity']},
    {'System': 'E2-B (recall=1.0)',
     'AUC': df_ref_r1.loc['mean','AUC'], 'Precision': df_ref_r1.loc['mean','Precision'],
     'Recall': df_ref_r1.loc['mean','Recall'], 'F1': df_ref_r1.loc['mean','F1'],
     'Specificity': df_ref_r1.loc['mean','Specificity']},
]).set_index('System')
table_comp.to_csv(PATHS.tables_dir / 'table_final_comparison.csv')
table_comp.round(4)

## 13. Error analysis  *(5.2.9)*

In [ ]:
skf = StratifiedKFold(n_splits=CONFIG.cv_n_splits, shuffle=True,
                      random_state=CONFIG.random_state)
scores_ref = {}
for u in CONFIG.umbral_range:
    dims_u = np.where(score_D > u)[0]
    if len(dims_u) < 2:
        continue
    scores_ref[u] = score_all_patients(patients, term_emb_raw, dims_u,
                                       top_n=CONFIG.top_n_tokens)

records = []
for fold, (tr, te) in enumerate(skf.split(labels_all, labels_all), 1):
    best_f1, best_u = -1, list(scores_ref)[0]
    for u, sc in scores_ref.items():
        p, r, _ = precision_recall_curve(labels_all[tr], sc[tr])
        f1 = 2*p*r/(p+r+1e-8)
        val = f1[:-1].max() if len(f1) > 1 else 0.0
        if val > best_f1:
            best_f1, best_u = val, u
    sc = scores_ref[best_u]
    p, r, thr = precision_recall_curve(labels_all[tr], sc[tr])
    f1 = 2*p*r/(p+r+1e-8)
    threshold = thr[min(np.argmax(f1[:-1]), len(thr)-1)] if len(thr) else sc[tr].mean()
    preds = (sc[te] >= threshold).astype(int)
    for i, idx in enumerate(te):
        pid = patients.ids[idx]
        y, yp = labels_all[idx], preds[i]
        kind = 'TP' if (y==1 and yp==1) else 'FP' if (y==0 and yp==1) else 'FN' if (y==1 and yp==0) else 'TN'
        records.append({'patient_id': pid, 'nhc': patients.nhc.get(pid, pid),
                        'label': int(y), 'pred': int(yp), 'score': float(sc[te][i]),
                        'type': kind, 'n_words': len(patients.words[pid]),
                        'fold': fold, 'tau': best_u})

df_pred = pd.DataFrame(records)
print('Confusion matrix (aggregated across folds):')
print(df_pred.groupby(['label','pred']).size().unstack(fill_value=0))
print('\nDocument length by prediction type:')
print(df_pred.groupby('type')['n_words'].agg(['mean','median','count']).round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(data=df_pred, x='type', y='n_words', order=['TP','TN','FP','FN'], ax=axes[0])
axes[0].set_title('Document length by prediction type')
axes[0].set_xlabel('Type'); axes[0].set_ylabel('Number of tokens')
sns.histplot(data=df_pred[df_pred['label']==1], x='score', hue='type', bins=15,
             palette={'TP':'#2ecc71','FN':'#e74c3c'}, ax=axes[1])
axes[1].set_title('Score distribution — asthma patients (TP vs FN)')
axes[1].set_xlabel('Score')
plt.tight_layout()
plt.savefig(PATHS.figures_dir / 'fig_error_analysis.png', dpi=150)
plt.show()
df_pred.to_csv(PATHS.tables_dir / 'table_predictions.csv', index=False)

## 14. Computational cost  *(5.2.10)*

In [ ]:
from asthma_e2.scoring import score_patient

dims_infer = np.where(score_D > 0.20)[0]
term_infer = term_emb_raw[:, dims_infer]
centroid_full = asthma_centroid(term_emb_raw)

_ = score_patient(patients.embeddings[patients.ids[0]], term_infer,
                  centroid_full, dims_infer)  # warm-up

times = []
for pid in tqdm(patients.ids, desc='Inference timing'):
    t0 = time.perf_counter()
    _ = score_patient(patients.embeddings[pid], term_infer, centroid_full,
                      dims_infer, top_n=CONFIG.top_n_tokens)
    times.append(time.perf_counter() - t0)

print(f'Graph construction   : {t_graph:.1f} s')
print(f'Embedding propagation: {t_prop:.1f} s')
print(f'Inference per patient: {np.mean(times)*1000:.1f} ± {np.std(times)*1000:.1f} ms')

table_cost = pd.DataFrame({
    'Stage': ['Graph construction', 'Embedding propagation',
              'Inference (mean per patient)'],
    'Time':  [f'{t_graph:.1f} s', f'{t_prop:.1f} s', f'{np.mean(times)*1000:.1f} ms'],
})
table_cost.to_csv(PATHS.tables_dir / 'table_cost.csv', index=False)
table_cost

---
### Output summary

| File | Section |
|------|---------|
| `table_1_graph.csv` | 5.2.1 Graph characterisation |
| `table_2_reference_pipeline.csv` | 5.2.2 Reference pipeline |
| `table_3_graph_ablation.csv` | 5.2.3 Graph ablation |
| `table_4_propagation_ablation.csv` | 5.2.4 Propagation ablation |
| `table_5_cs_ablation.csv` | 5.2.5 C×S criterion ablation |
| `table_6_subspace.csv` | 5.2.6 Discriminative subspace |
| `table_7_topn.csv` | 5.2.7 Lexical prefiltration |
| `table_final_comparison.csv` | 5.2.8 System comparison |
| `table_predictions.csv` | 5.2.9 Error analysis |
| `table_cost.csv` | 5.2.10 Computational cost |

All tables are written to `results/tables/` and figures to `results/figures/`.